# Imports

In [ ]:
!pip install streamlit pandas numpy scikit-learn matplotlib seaborn geopy

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sbn
import joblib

# Read dataframe

In [ ]:
airbnb = pd.read_csv('AB_NYC_2019.csv')

# Data exploration

In [ ]:
# Count of rows and columns
airbnb.shape

In [ ]:
# All the column names and their datatype
airbnb.info()

In [ ]:
# First 3 in the dataframe
airbnb.head(2)

In [ ]:
# 3 Random in dataframe
airbnb.sample(2)

In [ ]:
# The ranges, mean, deviation and quantiles of the numeric attributes
airbnb.describe()

# Data Cleaning

Our cleaning points:
* id
* insert room_type+neigbourhood for NaN name
* remove last_review (can't use it)
* fill reviews_per_month with mean groupby room_type+neighbourhood

In [ ]:
# Checking for NaN
airbnb.isna().sum()

In [ ]:
# Removing last_review
airbnb.drop(columns=['last_review'], inplace=True)

In [ ]:
# Fill Host_Name with "John Doe"
airbnb['host_name'] = airbnb['host_name'].fillna('John Doe')

In [ ]:
# Checking for NaN
airbnb.isna().sum()

# Feature Engineering

### Convert roomtype to numbers

In [ ]:
# Replacing room_types with numbers in a new column called "num_room_type"
airbnb['num_room_type'] = airbnb['room_type'].map({
    'Private room': 1,
    'Entire home/apt': 2,
    'Shared room': 3
    })

In [ ]:
!pip install geopy

### New column - distance to center

In [ ]:
# New column with the distance between airbnb and central new york (Time Square in Midtown Manhatten)
from geopy.distance import geodesic

# Constant variable (time square coordinates)
timesq_coord = (40.758896,-73.985130)

# Apply geodesic to airbnb coords
airbnb["dist_to_cent_km"] =  airbnb.apply(lambda x: geodesic(timesq_coord, (x["latitude"], x["longitude"])).km, axis=1)

In [ ]:
airbnb["dist_to_cent_km"]

# Analysis of variables

### Histogram (price)

In [ ]:
airbnb.hist()

In [ ]:
airbnb['price'].hist(bins=50)
plt.title("Distribution of Airbnb Prices")
plt.show()

### Average price by room type

In [ ]:
airbnb.groupby('room_type')['price'].mean().plot(kind='bar')
plt.title("Average Price by Room Type")
plt.show()

### Average price by neighbourhood group


In [ ]:
airbnb.groupby('neighbourhood_group')['price'].mean().plot(kind='bar')
plt.title("Average Price by Neighbourhood Group")
plt.show()

### Scatter plots

In [ ]:
airbnb.plot.scatter(x='number_of_reviews', y='price')
plt.title("Price vs Number of Reviews")
plt.show()

In [ ]:
# Plot regression of distance to centrum and price
plt.scatter(airbnb['dist_to_cent_km'], airbnb['price'], alpha=0.3)
plt.xlabel("Distance to Center")
plt.ylabel("Price")
plt.title("Distance vs Price")
plt.show()

### Outliers

In [ ]:
# First we want to check for outliers
airbnb.boxplot(column="price")

In [ ]:
# We want to improve the visualisation for our boxplot. The boxplot quantiles is almost not visible because most of the values
# are low and there are some really high prices in the dataset, therefore we use log to compress the values (especially the high values)
import numpy as np

airbnb["log_price"] = np.log1p(airbnb["price"])
airbnb.boxplot(column="log_price")

In [ ]:
# Here we are using IQR to remove outliers, q3-q1, we could obviously see some outliers in price in our previous boxplot.
# We are removing the outliers in the numeric values we want to continue using in ML.
from typing import List

def remove_outliers(df, data):

    q1 = data.quantile(0.25)
    q3 = data.quantile(0.75)

    IQR = q3 - q1

    # rows with outliers
    outliers = (data < (q1 - 1.5 * IQR)) | (data > (q3 + 1.5 * IQR))

    # find rows containing any outlier
    outlier_rows = outliers.any(axis=1)

    # drop them
    df = df.drop(df[outlier_rows].index)

    return df



#remove_outliers(airbnb, airbnb[['price','minimum_nights','number_of_reviews','availability_365','dist_to_cent_km']])
airbnb = remove_outliers(
    airbnb,
    airbnb[['price','minimum_nights','number_of_reviews','availability_365','dist_to_cent_km']]
)

In [ ]:
# Boxplot of price after removing outliers
airbnb.boxplot(column="price")

Removing outliers from price had a huge impact on the visualisation for the boxplot. The quantiles aren't compressed anymore and the quantiles + median is a lot more visible now.

### Correlation heatmap

In [ ]:
corr = airbnb[['price','minimum_nights','number_of_reviews','availability_365', 'dist_to_cent_km']].corr()
sbn.heatmap(corr, annot=True)
plt.title("Correlation Heatmap")
plt.show()

# Machine Learning

## Linear Regression (Supervised Learning)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

In [ ]:
# Feature og target

#Hot encoding
airbnb = pd.get_dummies(airbnb, columns=['neighbourhood_group'], drop_first=True)

X = airbnb[['dist_to_cent_km',
            'minimum_nights',
            'number_of_reviews',
            'availability_365',
            'num_room_type',
            'neighbourhood_group_Brooklyn',
            'neighbourhood_group_Queens',
            'neighbourhood_group_Staten Island']]
y = airbnb['price']

In [ ]:
# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
# Train model
model = LinearRegression()
model.fit(X_train, y_train)

In [ ]:
# Predictions
y_predicted = model.predict(X_test)
y_predicted

In [ ]:
# Evaluate model
print("MSE:", mean_squared_error(y_test, y_predicted))
print("R2:", r2_score(y_test, y_predicted))

## Cluster (Unsupervised Learning)

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from scipy.spatial.distance import cdist
from yellowbrick.cluster import SilhouetteVisualizer
from sklearn.cluster import KMeans
# We are choosing the K-means algorithm for our clustering and segmentation
X_cluster = airbnb[['dist_to_cent_km',
            'minimum_nights',
            'number_of_reviews',
            'availability_365', 'num_room_type', 'price']].values

# K-mean is distance based so we need to scale so all our variables is available
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cluster)

### Elbow method

In [ ]:
# Determine k by minimizing the distortion - 
# the sum of the squared distances between each observation vector and its centroid
distortions = []
K = range(2, 10)
for k in K:
    model = KMeans(n_clusters=k, n_init=10, random_state=42)
    model.fit(X_scaled)
    distortions.append(sum(np.min(cdist(X_scaled, model.cluster_centers_, 'euclidean'), axis=1)) / X_scaled.shape[0])
    # print all distortions to identify the lowest gain
print("Distortion: ", distortions)

In [ ]:
# Plotting the distortion to discover the elbow
plt.title('Elbow Method for Optimal K')
plt.plot(K, distortions, 'bx-')
plt.xlabel('K')
plt.ylabel('Distortion')
plt.show()

The elbow method didnt visualize any precise break to indicate how many clusters to use. We can see a little break at 3-4. When making the silhouette score i want to choose the amount of clusters by the highest score.

### Determin K by Silhouette Score

In [ ]:
scores = []
for k in K:
    model = KMeans(n_clusters=k, n_init=10, random_state=42)
    model.fit(X_scaled)
    score = silhouette_score(X_scaled, model.labels_)
    print(f'Number of clusters = {k}, Silhouette Score = {score:.3f}')
    scores.append(score)

# Plotting the score 
plt.plot(K, scores, 'bx-')
plt.xlabel('K')
plt.ylabel('Silhouette Score')
plt.title('Silhouette Score Method for Optimal K')
plt.show()

We wanted to look for the amount of clusters, mostly between 3-5, we do that by finding the highest silhouette score. The highest silhouette score between 2-9 is cluster number 4 with 0.247 which is going to be the amount we want to use.

### Train model

In [ ]:
# Optimal number of clusters K (seems to be 4)
num_clusters = 4

# Creating an instance of KMeans classifier
kmeans = KMeans(n_clusters=num_clusters, n_init=20, random_state=42)
# n_init: the algorithm will run n_init times with different cetroids and the best result of those will be taken

# Train the KMeans clustering model
kmeans.fit(X_scaled)

In [ ]:
# Saving cluster in labels in our airbnb dataframe
airbnb['cluster'] = kmeans.labels_

### Visualization

In [ ]:
# Firstly we want to look at a scatter plot of price and dist_to_cent_km with cluser labels
plt.scatter(airbnb["dist_to_cent_km"], airbnb["price"], c=airbnb["cluster"], cmap='viridis', s=50)
plt.xlabel('Distance to center')
plt.ylabel('Price')
plt.title('Airbnb Clusters')
plt.show()

When comparing price and distance to center with clusters, we can see yellow and green are more compact and defined with the dots compared to cyan and purple which is more spread out. To get a better visualization we want to use all 6 variables at once, to do that we need PCA.

In [ ]:
# To visualize all 6 variables we use PCA 
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

plt.scatter(X_pca[:,0], X_pca[:,1], c=airbnb['cluster'], cmap='viridis', s=50)
plt.title('Airbnb Clusters (PCA projection)')
plt.show()

The PCA visualization shows the clusters clearly grouped/segmented and compact but with some few overlaps.

### Validate Model

In [ ]:
k = 4
model = KMeans(n_clusters=k, n_init=10)
model.fit_predict(X)

In [ ]:
score = silhouette_score(X, model.labels_, metric='euclidean')
print('Silhouette Score: %.3f' % score)

When looking at the silhouette score we look at the range between -1 and 1. The score 0.689 is good and tells us our data is close and connected to their own cluster, the cluster is seperated and doesnt overlap eachother as much.
It's not perfect which shows that some places the cluster can overlap.

In [ ]:
# We also want to visualize our Silhouette score to see if the clusters are close or have distance to eachother
# Visualize the silhouette scores of all points
visualizer = SilhouetteVisualizer(kmeans, colors='yellowbrick')
visualizer.fit(X_scaled)
visualizer.show()

Interpretation: Each component of the figure represents one cluster with a horisontal bar chart of each cluster point. Our cluster 2 and 3 is more defined and dominates more compared to 0 and 1. Our average score is close to 0 which means our dots/data is close to the clusters boundary. Avg score 0.240 - Okay

### Train with new data

In [ ]:
X_cluster = airbnb[['dist_to_cent_km',
            'minimum_nights',
            'number_of_reviews',
            'availability_365', 'num_room_type', 'price']].values
# Defining 2 new airbnbs with Attributes: 
# dist_to_cent_km, minimum_nights, number_of_reviews, availability_365, num_room_type and price
test1 = np.array([[12.4, 1, 15, 300, 1, 750]]) # new airbnb 1 (close to center and high price)
test2 = np.array([[40, 3, 2, 365, 2, 200]]) # new airbnb 2 (distance to center but cheaper)

# Standardize with the same scaler used for training
scaled_test1 = scaler.transform(test1)
scaled_test2 = scaler.transform(test2)


# Predict cluster labels
cluster1 = kmeans.predict(scaled_test1)
cluster2 = kmeans.predict(scaled_test2)

print(cluster1, cluster2)

In [ ]:
# If we want some sort of supervised learning, we can categorize them by the mean 
# of each attribute and see if there are any tendencies between the attribute and their cluster.
numeric_cols = ['dist_to_cent_km',
            'minimum_nights',
            'number_of_reviews',
            'availability_365', 'num_room_type', 'price']  
cluster_summary = airbnb.groupby('cluster')[numeric_cols].mean()
print(cluster_summary)

By quick observation we can analyze cluster 3 is the more attracted one, the closest to the center, highest price, highest minimum nights and it's most likely and entire home/apartment.

### Location visualization using folium and our clusters

In [ ]:
import folium
from folium import plugins
from folium.plugins import MousePosition
from folium.plugins import MarkerCluster

In [ ]:
# create a map object and use it as background map, give central point and map style
# Location Timesquare
location = [40.758896,-73.985130]
mapa = folium.Map(location=location, tiles="OpenStreetMap", width="100%", height="100%", zoom_start=6)

In [ ]:
# see the coordinates of the mouse as you move it
MousePosition().add_to(mapa)

In [ ]:
# Using MarkerCluster to our mapa to show our coordinates with our own clusters, by using a loop
marker_cluster = MarkerCluster().add_to(mapa)

colors = ["blue", "green", "red", "purple"]

# Cluster 0 = blue
# Cluster 1 = green
# Cluster 2 = red
# Cluster 3 = purple

for _, row in airbnb.iterrows():
    folium.CircleMarker(
        location=[row["latitude"], row["longitude"]],
        radius=3,
        color=colors[row["cluster"]],
        fill=True
    ).add_to(marker_cluster)

In [ ]:
folium.Marker(
    location=location,
    popup="Times Square",
    icon=folium.Icon(color="black")
).add_to(mapa)

In [ ]:
mapa

In [ ]:
joblib.dump(kmeans, 'kmeans_model.pkl')
joblib.dump(scaler, 'scaler.pkl')

## Classification (Supervised Learning)

### Create labels

In [ ]:
labels = ['Cheap', 'Normal', 'Expensive', 'Very expensive']

airbnb['price_segment'] = pd.qcut(airbnb['price'], q=4, labels=labels)

### Define the independent variables

In [ ]:
X = airbnb[['dist_to_cent_km', 'minimum_nights','number_of_reviews','availability_365', 'num_room_type']]
  # We define the variable that we want to explore and is dependent
y = airbnb['price_segment']

### Train/Test split

In [ ]:
# We are splitting our Dataframe 80/20, 80% for training purposes and 20% for testing purposes
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=123, test_size=0.20)

### Model (Random Forest)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
# Creating the classification model
clf = RandomForestClassifier(random_state=123)
# random_state does so we get the same results every time we run it, if the given value is the same.

### Train Model

In [ ]:
clf.fit(X_train, y_train) 

### Predict

In [ ]:
y_pred = clf.predict(X_test)

### Validation/Accuracy of model

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
# Accuracy tells us how many of the prediction were correct
# We got 0.50 so it's 50%
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

Shows more detailed information about the performance of the model.
It shows how well a job the model does of predicting the price segments

In [ ]:
print("\nClassification report:")
print(classification_report(y_test, y_pred))

Shows how many airbnbs were placed correctly and incorrectly.
It gives us information about where it goes wrong

In [ ]:
print("\nConfusion matrix:")
print(confusion_matrix(y_test, y_pred))

In [ ]:
print(airbnb['price_segment'].value_counts())

In [ ]:
# Save random forest model
joblib.dump(clf, 'random_forest_model.pkl')

### Decision Tree

In [ ]:
# We are adding decision tree to observe and visualize accuracy if we only have one tree instead of multiple.
from sklearn.tree import DecisionTreeClassifier
dt = DecisionTreeClassifier(max_depth=4, random_state=123)

dt.fit(X_train, y_train)

y_pred_tree = dt.predict(X_test)

print("Decision Tree Accuracy:", accuracy_score(y_test, y_pred_tree))

In [ ]:
from sklearn.tree import plot_tree
import matplotlib.pyplot as plt
# Our plot shows when decision is being taken because of the value of the attribute.
plt.figure(figsize=(12,8))
plot_tree(dt, feature_names=X.columns, filled=True)
plt.show()

In [ ]:
#We also want to look into what variables had the most affect on making the price_segment.
importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': dt.feature_importances_
})

print(importance.sort_values(by="Importance", ascending=False))

## Polynomial Regression

In [ ]:
X = airbnb[['dist_to_cent_km',
            'minimum_nights',
            'number_of_reviews',
            'availability_365',
            'num_room_type',
            'neighbourhood_group_Brooklyn',
            'neighbourhood_group_Queens',
            'neighbourhood_group_Staten Island']]

y = airbnb['price']

### Splitting the data

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
#
poly_model = PolynomialFeatures(degree=3)
X_poly_train = poly_model.fit_transform(X_train)
X_poly_test = poly_model.transform(X_test)
pol_reg = LinearRegression()
pol_reg.fit(X_poly_train, y_train)

In [ ]:
y_pred_poly = pol_reg.predict(X_poly_test)

### Validation

In [ ]:
print("Polynomial R2:", r2_score(y_test, y_pred_poly))
print("Polynomial RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_poly)))

### Visualize

In [ ]:
# Visualizing the Polymonial Regression results
# Only using variable dist_to_cent_km  and price to visualization
X_plot = airbnb[['dist_to_cent_km']].values
y = airbnb['price'].values

### Polynomial transformation

In [ ]:
poly_model = PolynomialFeatures(degree=3)
X_poly = poly_model.fit_transform(X_plot)

### Fit model

In [ ]:
pol_reg = LinearRegression()
pol_reg.fit(X_poly, y)

In [ ]:
# Save poly model
joblib.dump(pol_reg, 'poly_reg_model.pkl')
joblib.dump(poly_model, 'poly_transformer.pkl') 

### Grid til smooth kurve

In [ ]:
X_grid = np.linspace(X_plot.min(), X_plot.max(), 100).reshape(-1,1)
X_grid_poly = poly_model.transform(X_grid)

### Plot

In [ ]:
plt.scatter(X_plot, y, color='red', alpha=0.3)

plt.plot(
    X_grid,
    pol_reg.predict(X_grid_poly),
    color='blue',
    linewidth=3
)

plt.xlabel("Distance to Times Square (km)")
plt.ylabel("Price")
plt.title("Polynomial Regression: Distance vs Price")

plt.show()

### Prediction

In [ ]:
new_distance = np.array([[2]])

new_distance_poly = poly_model.transform(new_distance)

predicted_price = pol_reg.predict(new_distance_poly)

print("Predicted price:", predicted_price)

airbnb[['price', "dist_to_cent_km"]].describe()

## P-value and Correlation

In [ ]:
from scipy.stats import pearsonr
corr, p_value = pearsonr(airbnb['dist_to_cent_km'], airbnb['price'])

print("Correlation:", corr)
print("P-value:", p_value)

The correlation value -0.389 is a moderate negative correlation between price and distance to center. The result shows the more distant the airbnb is, the lower the price will be. There are still a lot of other factors that affects the price. 

The P-value is extremely low, it's p_value < 0.05, which means that we reject h0.
Because the P-value is extremely low we can conclude the correlation between price and distance isn't a coincidence.